In [ ]:
import os
from pathlib import Path
# Resolve submission_partition_draft root from common launch locations.
def _resolve_partition_root():
    cwd = Path(os.getcwd()).resolve()
    for p in [cwd, *cwd.parents]:
        if (p / "primary_script").exists() and (p / "intermediate").exists():
            return p
        candidate = p / "technical_review" / "submission_partition_draft"
        if (candidate / "primary_script").exists():
            return candidate
    raise RuntimeError("Could not locate submission_partition_draft root")
REPO_ROOT = _resolve_partition_root()


In [ ]:
import os
from pathlib import Path
import numpy as np
import doubletdetection
import tarfile
import matplotlib
import matplotlib.pyplot as plt
import sys
import pandas as pd
import scanpy as sc
from scipy.io import mmwrite
import anndata
import torch
import scvi
import scrublet as scr
import re

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [ ]:
outdir = str(REPO_ROOT / 'intermediate' / 'pbmc')
os.makedirs(outdir, exist_ok=True)


In [ ]:
lanes = ['MM-' + str(x) for x in range(1,9)]

### DoubletDetection

In [ ]:
for i in range(0,len(lanes)):
    print('starting doubletdetection on: ' + lanes[i])
    adata_solo = sc.read_10x_h5(str(REPO_ROOT / 'primary_dependents' / 'cellranger_h5' / ('10872-' + lanes[i]) / 'filtered_feature_bc_matrix.h5'))
    adata_solo.var_names_make_unique()
    soc = pd.read_csv(str(REPO_ROOT / 'primary_dependents' / 'souporcell' / ('10872-' + lanes[i] + '_soc') / 'clusters.tsv'), sep='\t')
    singlet_bc = soc[soc['status'] == 'singlet']['barcode']
    tmp_adata = adata_solo[adata_solo.obs.index.isin(singlet_bc), :]
    clf = doubletdetection.BoostClassifier(
        n_iters=50,
        clustering_algorithm="leiden",
        standard_scaling=True,
        pseudocount=0.1,
        verbose = False, 
        n_jobs=40,
    )
    doublets = clf.fit(tmp_adata.X).predict(p_thresh=1e-16, voter_thresh=0.5)
    doublet_score = clf.doublet_score()
    tmp_adata.obs["DoubletDetection_DropletType"] = doublets
    tmp_adata.obs["doublet_score"] = doublet_score
    tmp_adata.obs["Barcode"] = tmp_adata.obs.index.values
    dd = tmp_adata.obs[["Barcode","DoubletDetection_DropletType","doublet_score"]]
    dd.DoubletDetection_DropletType = dd.DoubletDetection_DropletType.replace(1.0, "doublet")
    dd.DoubletDetection_DropletType = dd.DoubletDetection_DropletType.replace(0.0, "singlet")
    dd.to_csv(os.path.join(outdir,lanes[i] + '_DoubletDetection_doublet_prediction.tsv'), sep = "\t", index = False)

In [ ]:
for i in range(0,len(lanes)):
    res = pd.read_csv(os.path.join(outdir,lanes[i] + '_DoubletDetection_doublet_prediction.tsv'), sep = "\t")
    bc_repl = re.compile(r'[0-9]+$').search(lanes[i]).group()
    res['Barcode'] = [re.sub(pattern = r'[0-9]+$', repl = bc_repl, string = x) for x in res['Barcode']]
    res.to_csv(os.path.join(outdir,lanes[i] + '_DoubletDetection_doublet_prediction.tsv'), sep = "\t", index = False)

### Scrublet

In [ ]:
for i in range(0,len(lanes)):
    print('starting scrublet on: ' + lanes[i])
    adata_solo = sc.read_10x_h5(str(REPO_ROOT / 'primary_dependents' / 'cellranger_h5' / ('10872-' + lanes[i]) / 'filtered_feature_bc_matrix.h5'))
    adata_solo.var_names_make_unique()
    soc = pd.read_csv(str(REPO_ROOT / 'primary_dependents' / 'souporcell' / ('10872-' + lanes[i] + '_soc') / 'clusters.tsv'), sep='\t')
    singlet_bc = soc[soc['status'] == 'singlet']['barcode']
    tmp_adata = adata_solo[adata_solo.obs.index.isin(singlet_bc), :]
    dbl_rate = tmp_adata.X.shape[0]/1000 * 0.008
    scrub = scr.Scrublet(tmp_adata.X, expected_doublet_rate=dbl_rate, sim_doublet_ratio = 2)
    doublet_scores, predicted_doublets = scrub.scrub_doublets(min_counts=3,
                                                              min_cells=3,
                                                              min_gene_variability_pctl=85,
                                                              n_prin_comps=30)
    tmp_adata.obs["scrublet_DropletType"] = predicted_doublets
    tmp_adata.obs["scrublet_score"] = doublet_scores
    tmp_adata.obs["Barcode"] = tmp_adata.obs.index.values
    scrdf = tmp_adata.obs[["Barcode","scrublet_DropletType","scrublet_score"]]
    scrdf.scrublet_DropletType = scrdf.scrublet_DropletType.replace(True, "doublet")
    scrdf.scrublet_DropletType = scrdf.scrublet_DropletType.replace(False, "singlet")
    scrdf.to_csv(os.path.join(outdir,lanes[i] + '_scrublet_results.tsv'), sep = "\t", index = False)

In [ ]:
for i in range(0,len(lanes)):
    res = pd.read_csv(os.path.join(outdir,lanes[i] + '_scrublet_results.tsv'), sep = '\t')
    bc_repl = re.compile(r'[0-9]+$').search(lanes[i]).group()
    res['Barcode'] = [re.sub(pattern = r'[0-9]+$', repl = bc_repl, string = x) for x in res['Barcode']]
    res.to_csv(os.path.join(outdir,lanes[i] + '_scrublet_results.tsv'), sep = "\t", index = False)

### solo

In [ ]:
for i in range(0,len(lanes)):
    print('starting solo on: ' + lanes[i])
    adata_solo = sc.read_10x_h5(str(REPO_ROOT / 'primary_dependents' / 'cellranger_h5' / ('10872-' + lanes[i]) / 'filtered_feature_bc_matrix.h5'))
    adata_solo.var_names_make_unique()
    soc = pd.read_csv(str(REPO_ROOT / 'primary_dependents' / 'souporcell' / ('10872-' + lanes[i] + '_soc') / 'clusters.tsv'), sep='\t')
    singlet_bc = soc[soc['status'] == 'singlet']['barcode']
    tmp_adata = adata_solo[adata_solo.obs.index.isin(singlet_bc), :]
    sc.pp.filter_cells(tmp_adata, min_genes = 200)
    sc.pp.filter_genes(tmp_adata, min_cells = 10)
    sc.pp.highly_variable_genes(tmp_adata, n_top_genes = 5000, subset = True, flavor = 'seurat_v3')
    
    csv_data = pd.read_csv(str(REPO_ROOT / 'primary_dependents' / 'EXCLUDE_XY_TCR_IG.csv'), header = None)
    positions = np.where(tmp_adata.var_names.isin(csv_data[0]))[0]
    tmp_adata.var.iloc[positions, tmp_adata.var.columns.get_loc('highly_variable')] = False
    tmp_adata.var.loc[tmp_adata.var.index=="TRAV1-2", "highly_variable"] = True # force TRAV1-2 to be hvg; MAIT marker
    hvg_features = tmp_adata.var.index.values[tmp_adata.var['highly_variable']]
    tmp_adata = tmp_adata[:, tmp_adata.var_names.isin(hvg_features)].copy()
    
    scvi.model.SCVI.setup_anndata(tmp_adata)
    vae = scvi.model.SCVI(tmp_adata)
    vae.train()
    solo = scvi.external.SOLO.from_scvi_model(vae)
    solo.train()
    solo_df = solo.predict()
    solo_df['prediction'] = solo.predict(soft = False)
    solo_df = pd.DataFrame({'Barcode': solo_df.index, 
                            'solo_DropletType': solo_df['prediction'], 
                            'solo_doublet_score': solo_df['doublet'], 
                            'solo_singlet_score': solo_df['singlet']})
    solo_df.to_csv(os.path.join(outdir,lanes[i] + '_solo_results.tsv'), sep = "\t", index = False)
    print(str(np.sum(solo_df['solo_DropletType']=='doublet')) + ' doublets for ' + lanes[i])

In [ ]:
for i in range(0,len(lanes)):
    res = pd.read_csv(os.path.join(outdir,lanes[i] + '_solo_results.tsv'), sep = "\t")
    bc_repl = re.compile(r'[0-9]+$').search(lanes[i]).group()
    res['Barcode'] = [re.sub(pattern = r'[0-9]+$', repl = bc_repl, string = x) for x in res['Barcode']]
    res.to_csv(os.path.join(outdir,lanes[i] + '_solo_results.tsv'), sep = "\t", index = False)

### Read res

In [ ]:
import re
import glob

In [ ]:
matching_files = glob.glob(str(REPO_ROOT / 'intermediate' / 'pbmc' / '*'))
regex = re.compile(r"solo_results")
solo_match = [x for x in matching_files if regex.search(x)]
def sort_key(file):
    match = re.search(r"(?!MM137)MM-(\d+)", file)
    return int(match.group(1)) if match else float('inf') # Uses infinity for files that don't match
pbmc_solo_files = sorted(solo_match, key=sort_key)

In [ ]:
matching_files = glob.glob(str(REPO_ROOT / 'intermediate' / 'pbmc' / '*'))
regex = re.compile(r"DoubletDetection_doublet_prediction")
dd_match = [x for x in matching_files if regex.search(x)]
def sort_key(file):
    match = re.search(r"(?!MM137)MM-(\d+)", file)
    return int(match.group(1)) if match else float('inf') # Uses infinity for files that don't match
pbmc_dd_files = sorted(dd_match, key=sort_key)

In [ ]:
matching_files = glob.glob(str(REPO_ROOT / 'intermediate' / 'pbmc' / '*'))
regex = re.compile(r"scrublet_results")
scr_match = [x for x in matching_files if regex.search(x)]
def sort_key(file):
    match = re.search(r"(?!MM137)MM-(\d+)", file)
    return int(match.group(1)) if match else float('inf') # Uses infinity for files that don't match
pbmc_scr_files = sorted(scr_match, key=sort_key)

In [ ]:
matching_files = glob.glob(str(REPO_ROOT / 'intermediate' / 'pbmc' / '*'))
regex = re.compile(r"scds_results")
scds_match = [x for x in matching_files if regex.search(x)]
def sort_key(file):
    match = re.search(r"(?!MM137)MM-(\d+)", file)
    return int(match.group(1)) if match else float('inf') # Uses infinity for files that don't match
pbmc_scds_files = sorted(scds_match, key=sort_key)

In [ ]:
for i in range(8):
    solo_res = pd.read_csv(pbmc_solo_files[i], sep = '\t')
    solo_res.rename(columns={"Unnamed: 0": "Barcode"}, inplace=True)
    if pbmc_solo_files[i]==pbmc_solo_files[0]:
        solo_df = solo_res
    else:
        solo_df = pd.concat([solo_df, solo_res])
        solo_df = solo_df.reset_index(drop=True)
solo_df = solo_df[['Barcode','solo_singlet_score','solo_doublet_score','solo_DropletType']]

In [ ]:
for i in range(8):
    dd_res = pd.read_csv(pbmc_dd_files[i], sep = '\t')
    dd_res.rename(columns={"barcode_2": "Barcode"}, inplace=True)
    if pbmc_dd_files[i]==pbmc_dd_files[0]:
        dd_df = dd_res
    else:
        dd_df = pd.concat([dd_df, dd_res])
        dd_df = dd_df.reset_index(drop=True)
dd_df = dd_df[['Barcode','doublet_score','DoubletDetection_DropletType']]
dd_df.rename(columns={'doublet_score': 'DoubletDetection_score'}, inplace=True)

In [ ]:
for i in range(8):
    scr_res = pd.read_csv(pbmc_scr_files[i], sep = '\t')
    scr_res.rename(columns={"barcode_2": "Barcode"}, inplace=True)
    if pbmc_scr_files[i]==pbmc_scr_files[0]:
        scr_df = scr_res
    else:
        scr_df = pd.concat([scr_df, scr_res])
        scr_df = scr_df.reset_index(drop=True)
scr_df = scr_df[['Barcode','scrublet_score','scrublet_DropletType']]

In [ ]:
for i in range(8):
    scds_res = pd.read_csv(pbmc_scds_files[i], sep = '\t')
    scds_res.rename(columns={"barcode_2": "Barcode"}, inplace=True)
    if pbmc_scds_files[i]==pbmc_scds_files[0]:
        scds_df = scds_res
    else:
        scds_df = pd.concat([scds_df, scds_res])
        scds_df = scds_df.reset_index(drop=True)
scds_df = scds_df[['Barcode','cxds_score','cxds_DropletType']]

In [ ]:
solo_dd = pd.merge(left = solo_df, right = dd_df, how = 'inner', on = 'Barcode')
solo_dd_scr = pd.merge(left = solo_dd, right = scr_df, how = 'inner', on = 'Barcode')
all_df = pd.merge(left = solo_dd_scr, right = scds_df, how = 'inner', on = 'Barcode')
all_df.to_csv(os.path.join(outdir,'doublet_scores.csv'), index=False)